In [1]:
from contextlib import closing
from pathlib import Path

import numpy as np
from cdflib import cdfread, cdfepoch
from spacepy.pycdf import CDF
from viresclient import SwarmRequest

# Initialize SwarmRequest
SwarmRequest.COLLECTIONS.update(
    {'AEJ_LPL': ['SW_OPER_AEJ{}LPL_2F'.format(x) for x in 'ABC']}
)
SwarmRequest.PRODUCT_VARIABLES.update(
    {'AEJ_LPL': ['Latitude_QD', 'Longitude_QD', 'MLT_QD', 'J_NE', 'J_QD']}
)

# Download AEJALPL_2F data
request = SwarmRequest()
request.set_collection('SW_OPER_AEJALPL_2F')
request.set_products(measurements=SwarmRequest.PRODUCT_VARIABLES['AEJ_LPL'])
response = request.get_between('2016-09-01', '2016-09-02')
da = response.as_xarray()

# Get Timestamp from downloaded data
Timestamp = da['Timestamp'].values

[1/1] Processing:  100%|██████████|  [ Elapsed: 00:01, Remaining: 00:00 ]
      Downloading: 100%|██████████|  [ Elapsed: 00:00, Remaining: 00:00 ] (0.192MB)


In [2]:
response.sources

['SW_OPER_AEJALPL_2F_20160901T000000_20160901T235959_0102']

In [3]:
# Get t from cdf file
DATADIR = Path('~/data/L2/AEJxLPL_2F').expanduser()
product = DATADIR / f'{response.sources[0]}.cdf'

with closing(cdfread.CDF(product)) as cdf:
    display(cdf.varinq('t'))
    t = np.array(cdfepoch.encode(cdf.varget('t'), iso_8601=True), dtype='datetime64[ns]')

{'Variable': 't',
 'Num': 0,
 'Var_Type': 'zVariable',
 'Data_Type': 31,
 'Data_Type_Description': 'CDF_EPOCH',
 'Num_Elements': 1,
 'Num_Dims': 0,
 'Dim_Sizes': [],
 'Sparse': 'No_sparse',
 'Last_Rec': 2463,
 'Rec_Vary': True,
 'Dim_Vary': [],
 'Pad': array([2.14270973e-234]),
 'Compress': 6,
 'Block_Factor': 640}

In [7]:
Timestamp.min()

numpy.datetime64('2016-09-01T00:10:29.591570377')

In [8]:
t.min()

numpy.datetime64('2016-09-01T00:10:29.591000000')

In [6]:
np.testing.assert_array_equal(da['Timestamp'].values, t)

AssertionError: 
Arrays are not equal

Mismatched elements: 2464 / 2464 (100%)
Max absolute difference: 992304
 x: array(['2016-09-01T00:10:29.591570377', '2016-09-01T00:10:45.185351610',
       '2016-09-01T00:11:00.778828144', ...,
       '2016-09-01T23:53:29.758851528', '2016-09-01T23:53:45.358703136',
       '2016-09-01T23:54:00.959000111'], dtype='datetime64[ns]')
 y: array(['2016-09-01T00:10:29.591000000', '2016-09-01T00:10:45.185000000',
       '2016-09-01T00:11:00.778000000', ...,
       '2016-09-01T23:53:29.758000000', '2016-09-01T23:53:45.358000000',
       '2016-09-01T23:54:00.959000000'], dtype='datetime64[ns]')

In [3]:
from spacepy.pycdf import CDF


class CdfEpoch():
    """ CDF Epoch Time type conversions. """
    CDF_EPOCH_1970 = 62167219200000.0

    @classmethod
    def decode_ns(cls, cdf_raw_time):
        """ Convert CDF raw time to datetime64[ns]. """
        return np.rint(
            (np.asarray(cdf_raw_time) - cls.CDF_EPOCH_1970) * 1e6
        ).astype('datetime64[ns]')

    @classmethod
    def decode_us(cls, cdf_raw_time):
        """ Convert CDF raw time to datetime64[us]. """
        return np.rint(
            (np.asarray(cdf_raw_time) - cls.CDF_EPOCH_1970) * 1e3
        ).astype('datetime64[us]')

    @classmethod
    def decode_ms(cls, cdf_raw_time):
        """ Convert CDF raw time to datetime64[ms]. """
        return np.rint(
            (np.asarray(cdf_raw_time) - cls.CDF_EPOCH_1970)
        ).astype('datetime64[ms]')


In [39]:
# Get t from cdf file
DATADIR = Path('~/data/L2/AEJxLPL_2F').expanduser()
product = DATADIR / f'{response.sources[0]}.cdf'

print("------------------\n")
print("NASA cdf / spacepy.cdf\n")
with closing(CDF(str(product))) as cdf:
    print(cdf, "\n")
    cdf_epoch = cdf.raw_var('t')[...]
    print("CDF_EPOCH: %r \n" % cdf_epoch)
    print("nanoseconds: %r \n" % CdfEpoch.decode_ns(cdf_epoch))
    print("microseconds: %r \n" % CdfEpoch.decode_us(cdf_epoch))
    print("miliseconds: %r \n" % CdfEpoch.decode_ms(cdf_epoch))
    print("list of strings: %r \n" % cdfepoch.encode(cdf_epoch, iso_8601=True)[:10])
    print("string-to-nanoseconds: %r \n" % np.array(cdfepoch.encode(cdf_epoch, iso_8601=True), dtype='datetime64[ns]'))
    print("cdflib: %r \n" % cdf['t'][...])
    x = cdf['t'][...]

print("------------------\n")
print("cdflib\n")
with closing(cdfread.CDF(product)) as cdf:
    print(cdf, "\n")
    cdf_epoch = cdf.varget('t')
    print("CDF_EPOCH: %r \n" % cdf_epoch)
    print("nanoseconds: %r \n" % CdfEpoch.decode_ns(cdf_epoch))
    print("microseconds: %r \n" % CdfEpoch.decode_us(cdf_epoch))
    print("miliseconds: %r \n" % CdfEpoch.decode_ms(cdf_epoch))
    print("list of strings: %r \n" % cdfepoch.encode(cdf_epoch, iso_8601=True)[:10])
    print("string-to-nanoseconds: %r \n" % np.array(cdfepoch.encode(cdf_epoch, iso_8601=True), dtype='datetime64[s]'))

------------------

NASA cdf / spacepy.cdf

Confidence: CDF_DOUBLE [62]
J: CDF_DOUBLE [2464, 2]
J_QD: CDF_DOUBLE [2464]
Latitude: CDF_DOUBLE [2464]
Latitude_QD: CDF_DOUBLE [2464]
Longitude: CDF_DOUBLE [2464]
Longitude_QD: CDF_DOUBLE [2464]
MLT: CDF_DOUBLE [2464]
RMS_misfit: CDF_DOUBLE [62]
t: CDF_EPOCH [2464]
t_qual: CDF_EPOCH [62] 

CDF_EPOCH: array([6.36399078e+13, 6.36399078e+13, 6.36399079e+13, ...,
       6.36399932e+13, 6.36399932e+13, 6.36399932e+13]) 

nanoseconds: array(['2016-09-01T00:10:29.591570432', '2016-09-01T00:10:45.185351680',
       '2016-09-01T00:11:00.778828032', ...,
       '2016-09-01T23:53:29.758851584', '2016-09-01T23:53:45.358703104',
       '2016-09-01T23:54:00.959000064'], dtype='datetime64[ns]') 

microseconds: array(['2016-09-01T00:10:29.591570', '2016-09-01T00:10:45.185352',
       '2016-09-01T00:11:00.778828', ..., '2016-09-01T23:53:29.758852',
       '2016-09-01T23:53:45.358703', '2016-09-01T23:54:00.959000'],
      dtype='datetime64[us]') 

miliseconds